# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa  
This notebook provides a step-by-step guide for loading, exploring, and performing basic analysis on the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. We will use the Croissant schema and reference all entities (record sets, fields, columns) by their `@id` fields to ensure clarity and reproducibility.

## Dataset Source
* **Citation:** Mugotitsa, B., Maina, D., Owoko, H., Akinyi, L., Greenfield, J., Todd, J., Kavu, T. and Kiragga, A. 2026.
* **URL:** https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
* **Description:** The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` is installed!pip install mlcroissant

## 1. Data Loading  
Load Croissant dataset metadata and display general dataset info.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the Croissant schema URLcroissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"
# Load the dataset metadata (schema and inferences)dataset = mlc.Dataset(croissant_url)# mlcroissant returns a metadata object, not a dict -- use as an objectmeta = dataset.metadataprint(f"Dataset Title: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}")

## 2. Data Overview  
Review available record sets and their field structure and accessible `@id` values.

In [ ]:
# List all record set `@id`s and the field `@id`s for each record set.record_sets = list(dataset.record_sets)print("Found record sets (by @id):")for rec in record_sets:    print(f"- {rec['@id']}")    if 'fields' in rec:        print("  Fields (by @id):")        for field in rec['fields']:            print(f"    - {field['@id']} ({field.get('name','')})")    print()

## 3. Data Extraction  
Load records from each record set into a pandas DataFrame. All record and field selections are done by `@id`.

> **Note:** For this dataset, most survey data is in a record set with an `@id` typically related to the main CSV file, e.g. `cr:Kilifi_MH_Survey`. Use the output above to identify the exact `@id`.

In [ ]:
# Choose the main survey record set by @id (update to match output above; example below)# We'll collect all dataframes in a dict for further analysis.main_record_set_id = Nonefor rec in record_sets:    # Heuristic: choose first record set that looks like a main table    if 'kilifi' in rec['@id'].lower() or 'main' in rec['@id'].lower() or 'mh' in rec['@id'].lower() or 'survey' in rec['@id'].lower():        main_record_set_id = rec['@id']        breakif main_record_set_id is None and record_sets:    main_record_set_id = record_sets[0]['@id']print(f"Using main record set: {main_record_set_id}")
dataframes = {}for rec in record_sets:    rec_id = rec['@id']    records = list(dataset.records(record_set=rec_id))    dataframes[rec_id] = pd.DataFrame(records)
df = dataframes.get(main_record_set_id)print("Available columns (field @id):")print(list(df.columns))df.head()

## 4. Exploratory Data Analysis (EDA)  
We will select a numeric field by its `@id` for demonstration (for example, `@id: cr:gad7_total`). Update variables below with real `@id`s as previewed above.

- Filter records where GAD-7 (anxiety) score > 10 (clinically meaningful threshold)
- Normalize the GAD-7 score
- Group by a demographic categorical field (e.g. `@id: cr:gender`) to compute mean scores


In [ ]:
# Example field IDs -- update as needed to match your schema (from previous outputs)numeric_field_id = Nonegroup_field_id = None
# Auto-discover likely numeric and categorical fields for demonstration purposesfor c in df.columns:
    cl = c.lower()    if numeric_field_id is None and (('gad7' in cl or 'phq9' in cl or 'score' in cl) and 'total' in cl):        numeric_field_id = c    if group_field_id is None and (cl in ['cr:gender', 'cr:sex', 'cr:marital_status', 'cr:education']):        group_field_id = cif numeric_field_id is None:    # Fallback: pick first integer/float-like column    for c in df.columns:        if pd.api.types.is_numeric_dtype(df[c]):            numeric_field_id = c            breakif group_field_id is None:    group_field_id = 'cr:gender' if 'cr:gender' in df.columns else df.columns[0]
print(f"Numeric field for demo: {numeric_field_id}")print(f"Group-by field for demo: {group_field_id}")
# Drop missing values in the chosen field for demoif numeric_field_id in df.columns:    demo_field = df[numeric_field_id].dropna()    try:        threshold = 10        filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()        print(f"Filtered records: {len(filtered_df)} with {numeric_field_id} > {threshold}")        # Normalize        filtered_df[f"{numeric_field_id}_normalized"] = (            filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())        # Group (if field available)        if group_field_id in filtered_df.columns:            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")            print(grouped_df.head())    except Exception as e:        print(f"Could not process field {numeric_field_id}: {e}")else:    print(f"Field {numeric_field_id} not found.")

## 5. Visualization  
Let's visualize the distribution of the selected numeric field (GAD-7 score), and show mean scores by demographic group. Update field `@id`s as required based on the previous output.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns
# Plot distribution of the numeric fieldif numeric_field_id in df.columns:    plt.figure(figsize=(8,4))    sns.histplot(df[numeric_field_id].dropna().astype(float), kde=True, bins=20)    plt.title(f"Distribution of {numeric_field_id}")    plt.xlabel(numeric_field_id)    plt.ylabel("Count")    plt.show()    # Plot group means if group field is available    if group_field_id in df.columns:        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()        plt.figure(figsize=(8,4))        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means)        plt.title(f"Mean {numeric_field_id} by {group_field_id}")        plt.xlabel(group_field_id)        plt.ylabel(f"Mean {numeric_field_id}")        plt.show()

## 6. Conclusion

- We successfully loaded the Kilifi County mental health survey using the standardized Croissant schema and the `mlcroissant` library, referencing all fields and record sets by their `@id`s.
- The data contains rich demographic and psychological indicator variables including GAD-7 (anxiety), PHQ-9 (depression), and PSQ (perceived stress) scores.
- We filtered and analyzed records by common thresholds, normalized values, and visualized both the distribution of one key metric and its relationship with demographic variables.
- The FAIR² approach to metadata ensures clarity and reproducibility for downstream AI/ML workflows.

Further steps would include modeling, advanced EDA, or harmonizing across similar clinical datasets. For more information see [the Croissant specification](https://mlcommons.org/croissant/).